# 📘 Notebook 3: Gradient Vectors — Definition, Geometric Meaning & Steepest Ascent

**Week 2 — Calculus, Optimization & Probability Theory**

**Difficulty:** ⭐⭐ (Beginner-Friendly)

---

## 🏠 Why Does This Matter?

Your house price model has two weights: `w1` (sqft) and `w2` (bedrooms).  
Notebook 1 gave you `∂L/∂w1` and `∂L/∂w2` separately.

But what if we packed these into a single arrow — a vector that says:
> "Move in **this combined direction** to make the loss go up the fastest"

That arrow is called the **gradient vector** `∇L`.  
To **reduce** the loss, we move in the **opposite direction**: `−∇L`.  
This is the entire idea of gradient descent, made geometric.

---

## 📚 Prerequisites
- Notebook 1 (partial derivatives)
- Vectors and vector operations (Week 1)

---

## Part 1: What is a Gradient Vector?

### Plain English First

Imagine you are hiking and you want to know: **"Which direction is steepest uphill right now?"**

If you're on a 2D map (x and y coordinates), you can feel:
- The slope going **east-west** (partial derivative in x)
- The slope going **north-south** (partial derivative in y)

Combine these two into a 2D arrow. That arrow points in the direction of **steepest ascent**.  
Its **length** tells you how steep the hill is.

That arrow is the **gradient**.

### Mathematical Definition

For a function `f(x₁, x₂, ..., xₙ)`, the gradient is:

$$\nabla f = \begin{bmatrix} \partial f/\partial x_1 \\ \partial f/\partial x_2 \\ \vdots \\ \partial f/\partial x_n \end{bmatrix}$$

Plain English: **stack all the partial derivatives into a single column vector**.

For our house price loss `L(w1, w2) = (w1 − 2)² + (w2 + 1)²`:

$$\nabla L = \begin{bmatrix} \partial L/\partial w_1 \\ \partial L/\partial w_2 \end{bmatrix} = \begin{bmatrix} 2(w_1 - 2) \\ 2(w_2 + 1) \end{bmatrix}$$

### Key Properties (no formulas yet — just ideas)

1. `∇L` always points **uphill** (direction of steepest ascent)
2. `−∇L` points **downhill** (direction of steepest descent — what we use in GD!)
3. `‖∇L‖` (length of the gradient) = how steep the slope is
4. When `∇L = 0` (zero vector), the function is at a **flat point** — minimum, maximum, or saddle

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────
# House price loss: L(w1, w2) = (w1 - 2)² + (w2 + 1)²
# Minimum at (w1=2, w2=-1) — the "true" best weights
#
# Gradient:
#   ∂L/∂w1 = 2(w1 - 2)    ← 0 when w1 = 2
#   ∂L/∂w2 = 2(w2 + 1)    ← 0 when w2 = -1
# ─────────────────────────────────────────────────────────────────

def loss(w1, w2):
    """Simplified house price loss."""
    return (w1 - 2)**2 + (w2 + 1)**2

def gradient(w1, w2):
    """
    Gradient of the loss. Returns a 2D vector [∂L/∂w1, ∂L/∂w2].
    This vector points in the direction of steepest ASCENT.
    """
    dL_dw1 = 2 * (w1 - 2)    # partial derivative w.r.t. w1
    dL_dw2 = 2 * (w2 + 1)    # partial derivative w.r.t. w2
    return np.array([dL_dw1, dL_dw2])   # package as a 2D vector


# ─── Build the loss surface for plotting ──────────────────────────
w1_range  = np.linspace(-1, 5, 200)
w2_range  = np.linspace(-4, 2, 200)
W1, W2    = np.meshgrid(w1_range, w2_range)   # 2D grid of (w1, w2) pairs
Z         = loss(W1, W2)                       # loss at every grid point

# ─── Coarser grid for the gradient arrow overlay ──────────────────
# (We use fewer points so arrows don't overlap)
w1g = np.linspace(-1, 5, 14)     # 14 points along w1 axis
w2g = np.linspace(-4, 2, 14)     # 14 points along w2 axis
W1g, W2g = np.meshgrid(w1g, w2g) # coarse grid

# Compute gradient at each coarse grid point
Gw1 = np.zeros_like(W1g)    # will hold ∂L/∂w1 at each point
Gw2 = np.zeros_like(W2g)    # will hold ∂L/∂w2 at each point
for i in range(W1g.shape[0]):
    for j in range(W1g.shape[1]):
        g = gradient(W1g[i,j], W2g[i,j])   # compute gradient at this point
        Gw1[i,j] = g[0]                     # store w1 component
        Gw2[i,j] = g[1]                     # store w2 component

# Normalize arrows to unit length (so all arrows same size, just show direction)
mag      = np.sqrt(Gw1**2 + Gw2**2) + 1e-10   # magnitude, avoid div-by-zero
U_ascent = Gw1 / mag    # normalized direction component (w1)
V_ascent = Gw2 / mag    # normalized direction component (w2)

# ─── Plot: gradient (ascent) vs negative gradient (descent) ──────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (U, V), arrow_color, title_desc in [
    (axes[0], (U_ascent,  V_ascent),  'white',  '∇L (Gradient = UPHILL direction)'),
    (axes[1], (-U_ascent,-V_ascent),  'cyan',   '−∇L (Negative Gradient = DOWNHILL direction)'),
]:
    # Draw the loss surface as a colored contour map
    cf = ax.contourf(W1, W2, Z, levels=25, cmap='hot_r', alpha=0.75)
    ax.contour(W1, W2, Z, levels=10, colors='white', alpha=0.25, linewidths=0.5)  # contour lines
    plt.colorbar(cf, ax=ax, label='Loss L(w1, w2)')

    # Draw the arrows
    ax.quiver(W1g, W2g, U, V,
              color=arrow_color, alpha=0.8, scale=22,
              headwidth=4, headlength=5)

    # Mark the minimum (optimal weights)
    ax.plot(2, -1, 'lime', marker='*', markersize=20,
            label='Optimal weights (w1=2, w2=-1)', zorder=10)
    ax.set_xlabel('w₁  (sqft weight)', fontsize=11)
    ax.set_ylabel('w₂  (bedroom weight)', fontsize=11)
    ax.set_title(f'{title_desc}\nArrows show direction; colors show loss magnitude', fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle('Gradient vs Negative Gradient on the House Price Loss Surface',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Left  (white arrows): Gradient ∇L → points AWAY from the minimum (uphill)")
print("Right (cyan arrows):  Negative gradient −∇L → points TOWARD the minimum (downhill)")
print("Gradient descent follows the cyan arrows → eventually reaches the minimum!")

## Part 2: Why Gradient Points Uphill — The Math Behind It

### Plain English First

Given a direction `d` (a unit vector), the **directional derivative** tells you:  
> "If I walk in direction `d`, how fast does the loss change?"

$$D_d L = \nabla L \cdot d = \|\nabla L\| \cos(\theta)$$

where `θ` is the angle between `∇L` and `d`.

- **`θ = 0°`** (walking with the gradient): `cos(0)=1` → maximum increase → **steepest ascent**
- **`θ = 180°`** (walking against the gradient): `cos(180°)=-1` → maximum decrease → **steepest descent**
- **`θ = 90°`** (walking perpendicular): `cos(90°)=0` → **no change** → moving along a contour line!

**Key insight:** Contour lines (lines of equal loss) are always **perpendicular to the gradient**.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Directional derivatives: in which direction does the loss change fastest?
# Test at point (w1=0, w2=0) — far from the optimum (2, -1)
# ─────────────────────────────────────────────────────────────────

w_point = np.array([0.0, 0.0])         # current weights
g       = gradient(*w_point)           # gradient at this point

print(f"At w = {w_point}:")
print(f"  Gradient ∇L = {g}")
print(f"  Gradient magnitude ‖∇L‖ = {np.linalg.norm(g):.4f}")
print()
print("Directional derivative = ∇L · direction (dot product)")
print("Tells us: if we move in this direction, how fast does the loss change?")
print()

# Test 8 compass directions as unit vectors
angles_deg = np.arange(0, 360, 45)     # 0°, 45°, 90°, ..., 315°
print(f"{'Direction':25} | {'Angle':8} | {'Directional Derivative':22} | {'Loss changes?'}")
print("-" * 80)

for angle in angles_deg:
    theta   = np.deg2rad(angle)                         # convert to radians
    d       = np.array([np.cos(theta), np.sin(theta)])  # unit direction vector
    dir_d   = np.dot(g, d)                              # directional derivative = ∇L · d
    if dir_d > 1.0:   change = "↑ Loss INCREASES (uphill)"
    elif dir_d < -1.0: change = "↓ Loss DECREASES (downhill)"
    else:              change = "→ Loss barely changes"
    # Direction name
    names = ['East','NE','North','NW','West','SW','South','SE']
    name = names[list(angles_deg).index(angle)]
    print(f"  {name:8} [{d[0]:6.3f},{d[1]:6.3f}]  | {angle:6}°   | {dir_d:22.4f} | {change}")

print()
g_dir = g / np.linalg.norm(g)   # normalize gradient to unit vector
print(f"Gradient direction (steepest ascent):  {g_dir}")
print(f"Negative gradient (steepest descent): {-g_dir}  ← this is the direction to move!")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Visualizing gradient perpendicularity to contour lines
# Key geometric insight: gradient arrows always cross contours at 90°
# ─────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(9, 8))

# Draw contour lines (lines of equal loss)
cs = ax.contour(W1, W2, Z,
                levels=[0.5, 1, 2, 4, 7, 11, 16],   # specific loss values to show
                colors='royalblue', linewidths=2)
ax.clabel(cs, inline=True, fontsize=9, fmt='L=%.1f')  # label each contour with its loss value

# Draw gradient arrows at selected points
test_points = [(0.0, 0.0), (1.0, -0.5), (3.5, 0.5), (-0.5, -2.5), (3.0, -3.0), (0.5, 1.5)]

for (p1, p2) in test_points:
    g = gradient(p1, p2)                      # gradient at this point
    g_norm = g / np.linalg.norm(g)            # normalize: show direction only
    ax.annotate('', xy=(p1 + g_norm[0]*0.4, p2 + g_norm[1]*0.4),   # arrow tip
                xytext=(p1, p2),                                       # arrow base
                arrowprops=dict(arrowstyle='->', color='tomato', lw=2.5))
    ax.plot(p1, p2, 'o', color='tomato', markersize=7, zorder=5)     # dot at base

ax.plot(2, -1, 'g*', markersize=20, label='Optimal weights (∇L = 0 here)', zorder=10)
ax.set_xlabel('w₁  (sqft weight)', fontsize=12)
ax.set_ylabel('w₂  (bedroom weight)', fontsize=12)
ax.set_title('Gradient Vectors (red) are Always Perpendicular to Contour Lines (blue)\n'
             'House Price Loss Surface', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(-1, 5); ax.set_ylim(-4, 2)

# Add perpendicularity annotation
ax.text(3.8, 1.5, '90° angle\nbetween gradient\nand contour', fontsize=9,
        color='darkred', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print("Notice: every red arrow is perpendicular to the blue contour lines.")
print("This is a universal geometric property of gradients.")

## Part 3: The Gradient in a Real ML Context

### Plain English First

In a real house price regression model with 2 features and N training examples,  
the MSE loss is:

$$L(\mathbf{w}) = \frac{1}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)^2 = \frac{1}{N} \|X\mathbf{w} - \mathbf{y}\|^2$$

The gradient is:

$$\nabla_\mathbf{w} L = \frac{2}{N} X^\top (X\mathbf{w} - \mathbf{y})$$

Plain English: **"How does each weight's change propagate through all predictions and affect the total loss?"**

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Real house price dataset: 50 houses, 2 features
# Features: (sqft_normalized, bedrooms_normalized)
# Target: price_normalized
# ─────────────────────────────────────────────────────────────────

np.random.seed(0)
N = 50    # number of houses in our dataset

# Generate synthetic house data
X_data  = np.random.randn(N, 2)              # shape: (50, 2)  — 50 houses, 2 features
w_true  = np.array([2.0, -1.0])             # true weights: sqft matters more, bedrooms negatively
y_data  = X_data @ w_true + 0.1 * np.random.randn(N)   # true prices with small noise

def mse_loss(w):
    """
    MSE loss for the full dataset.
    w : weight vector, shape (2,)
    Returns a scalar loss value.
    """
    predictions = X_data @ w      # predicted prices for all 50 houses: (50,)
    residuals   = predictions - y_data   # prediction errors: (50,)
    return np.mean(residuals**2)          # mean squared error: scalar

def mse_gradient(w):
    """
    Gradient of MSE loss: ∇L = (2/N) * X.T @ (Xw - y)
    w : weight vector, shape (2,)
    Returns gradient vector, shape (2,).

    Derivation: L = (1/N)||Xw - y||²
      ∂L/∂w = (2/N) * X.T @ (Xw - y)
    Each component i: ∂L/∂wi = (2/N) Σⱼ (xⱼ·w - yⱼ) * xⱼᵢ
    """
    predictions = X_data @ w              # shape: (N,)
    residuals   = predictions - y_data    # shape: (N,)
    return 2 * X_data.T @ residuals / N  # shape: (2,) — one gradient per weight

# ─── Check the gradient at a few starting points ──────────────────
print("MSE Gradient for House Price Regression")
print(f"True weights: {w_true}")
print()

test_weights = [np.array([0.0, 0.0]),
                np.array([1.0, 0.5]),
                w_true]

eps = 1e-5
for w in test_weights:
    g_anal = mse_gradient(w)
    # Numerical check
    g_num0 = (mse_loss(w + [eps,0]) - mse_loss(w - [eps,0])) / (2*eps)
    g_num1 = (mse_loss(w + [0,eps]) - mse_loss(w - [0,eps])) / (2*eps)
    g_num  = np.array([g_num0, g_num1])
    print(f"w = {w}:")
    print(f"  Loss      = {mse_loss(w):.6f}")
    print(f"  ∇L (anal) = {g_anal}")
    print(f"  ∇L (num)  = {g_num}  (error = {np.max(np.abs(g_anal-g_num)):.2e})")
    print(f"  ‖∇L‖      = {np.linalg.norm(g_anal):.4f}  ← how steep the landscape is here")
    print()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Visualize the MSE gradient field over the weight space
# ─────────────────────────────────────────────────────────────────

# Build loss surface
w1_range = np.linspace(-1, 4, 150)
w2_range = np.linspace(-3, 2, 150)
W1g, W2g = np.meshgrid(w1_range, w2_range)
Z_mse = np.array([[mse_loss(np.array([w1v, w2v]))
                   for w1v in w1_range]
                  for w2v in w2_range])    # compute loss at every grid point

# Coarse gradient field
w1c = np.linspace(-1, 4, 12)
w2c = np.linspace(-3, 2, 12)
W1c, W2c = np.meshgrid(w1c, w2c)
U = np.zeros_like(W1c)    # ∂L/∂w1 direction component
V = np.zeros_like(W2c)    # ∂L/∂w2 direction component
for i in range(W1c.shape[0]):
    for j in range(W1c.shape[1]):
        g = mse_gradient(np.array([W1c[i,j], W2c[i,j]]))
        n = np.linalg.norm(g) + 1e-10
        U[i,j] = g[0] / n    # normalize: show direction, not magnitude
        V[i,j] = g[1] / n

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, (u, v), color, title_extra in [
    (axes[0], (U, V),   'white', 'Gradient ∇L (uphill)'),
    (axes[1], (-U, -V), 'cyan',  '−∇L (downhill = descent direction)'),
]:
    cf = ax.contourf(W1g, W2g, Z_mse, levels=25, cmap='hot_r')
    plt.colorbar(cf, ax=ax, label='MSE Loss')
    ax.quiver(W1c, W2c, u, v, color=color, alpha=0.75, scale=22, headwidth=4)
    ax.plot(w_true[0], w_true[1], 'b*', markersize=18, label=f'True weights {w_true}', zorder=10)
    ax.set_xlabel('w₁ (sqft weight)'); ax.set_ylabel('w₂ (bedroom weight)')
    ax.set_title(f'{title_extra}\non MSE loss surface', fontsize=12)
    ax.legend(fontsize=9)

plt.suptitle('MSE Gradient Field: Real House Price Regression (50 houses)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Following −∇L (cyan arrows) leads exactly to the true weights (blue star).")
print("This is gradient descent — always following the downhill direction.")

---

## ✅ Self-Check Questions

**1.** The gradient at your current weights is `∇L = [4.0, -2.0]`.  
   - Which weight (w1 or w2) is causing more of the loss increase?
   - In which direction should you move w2 to reduce the loss?

**2.** What does `‖∇L‖ = 0` tell you about the loss surface at that point?

**3.** You're on a contour line (a line of equal loss). In which direction relative  
   to the gradient vector should you walk to stay on the same contour?

**4.** Your model has 1 million weights. How many numbers are in its gradient vector `∇L`?

**5.** The directional derivative `D_d L = ∇L · d`. If `∇L = [3, 4]` and  
   `d = [0.6, 0.8]` (a unit vector), what is the directional derivative?

<details>
<summary>Click to see answers</summary>

1. **w1 is causing more** (|4| > |-2|). **w2 should be increased** — the derivative is -2 (negative), meaning increasing w2 reduces the loss.

2. `‖∇L‖ = 0` means the gradient is the zero vector — we are at a **critical point** (minimum, maximum, or saddle point). Gradient descent has converged.

3. Walk **perpendicular** to the gradient (at 90°) to stay on a contour line.

4. **1 million numbers** — one per weight. The gradient has the same shape as the parameter vector.

5. `D_d L = [3,4]·[0.6,0.8] = 3×0.6 + 4×0.8 = 1.8 + 3.2 = 5.0`.

</details>

---

## 📌 Summary

| Concept | One-line meaning | House price analogy |
|---------|-----------------|---------------------|
| **Gradient ∇L** | Vector of all partial derivatives | Combined direction of steepest weight adjustment |
| **−∇L** | Negative gradient | Direction to move weights to reduce price error |
| **‖∇L‖** | Gradient magnitude | How much the loss is changing — "how steep is the hill" |
| **∇L = 0** | Flat point | Optimal weights found (or saddle point) |
| **Contour ⊥ ∇L** | Perpendicularity | Paths of equal error are perpendicular to improvement direction |

**➡️ Next Notebook:** Gradient Descent — the algorithm that follows `−∇L` step by step to find the optimal weights.